In [0]:
from pyspark.sql import functions as F

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("catalog_name", "retail_lakehouse")
dbutils.widgets.text("pipeline_run_id", "")

In [0]:
import uuid

environment = dbutils.widgets.get("environment").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()

if not pipeline_run_id:
    pipeline_run_id = str(uuid.uuid4())

In [0]:
sales_df = spark.table(
    f"{catalog_name}.silver.sales_transactions"
)

products_df = spark.table(
    f"{catalog_name}.silver.products"
)

customers_df = spark.table(
    f"{catalog_name}.silver.customers"
)

stores_df = spark.table(
    f"{catalog_name}.silver.stores"
)

promotions_df = spark.table(
    f"{catalog_name}.silver.promotions"
)

In [0]:
#Selecting only required dimension columns:

product_dim_df = products_df.select(
    "product_id",
    "product_name",
    "category",
    "brand",
    "cost_price",
    "product_status"
)

customer_dim_df = customers_df.select(
    "customer_id",
    "customer_name",
    "age",
    "customer_age_group",
    "city",
    "state",
    "region",
    "loyalty_tier"
)

store_dim_df = stores_df.select(
    "store_id",
    F.col("store_name"),
    F.col("city").alias("store_city"),
    F.col("state").alias("store_state"),
    F.col("region").alias("store_region"),
    "store_size"
)

promotion_dim_df = promotions_df.select(
    "promotion_id",
    "promotion_name",
    F.col("category").alias("promotion_category"),
    "discount_type",
    "discount_value",
    "promotion_status"
)

In [0]:
sales_enriched_df = (
    sales_df.alias("sales")
    .join(
        product_dim_df.alias("product"),
        on="product_id",
        how="left"
    )
    .join(
        customer_dim_df.alias("customer"),
        on="customer_id",
        how="left"
    )
    .join(
        store_dim_df.alias("store"),
        on="store_id",
        how="left"
    )
    .join(
        promotion_dim_df.alias("promotion"),
        on="promotion_id",
        how="left"
    )
)

In [0]:
#Adding enrichment flags and profit:

sales_enriched_df = (
    sales_enriched_df
    .withColumn(
        "product_match_flag",
        F.when(
            F.col("product_name").isNotNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "customer_match_flag",
        F.when(
            F.col("customer_name").isNotNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "store_match_flag",
        F.when(
            F.col("store_name").isNotNull(),
            1
        ).otherwise(0)
    )
    .withColumn(
        "estimated_cost_amount",
        F.col("quantity") * F.col("cost_price")
    )
    .withColumn(
        "gross_profit_amount",
        F.col("recognized_sales_amount")
        - F.col("estimated_cost_amount")
    )
    .withColumn(
        "_enriched_timestamp",
        F.current_timestamp()
    )
)

In [0]:
enriched_table = (
    "retail_lakehouse.silver.sales_enriched"
)

(
    sales_enriched_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(enriched_table)
)

In [0]:
#Validating unmatched records:

sales_enriched_df.select(
    F.sum(
        F.when(F.col("product_match_flag") == 0, 1)
        .otherwise(0)
    ).alias("unmatched_products"),
    F.sum(
        F.when(F.col("customer_match_flag") == 0, 1)
        .otherwise(0)
    ).alias("unmatched_customers"),
    F.sum(
        F.when(F.col("store_match_flag") == 0, 1)
        .otherwise(0)
    ).alias("unmatched_stores")
).show()